# Preprocessing for the baseline condition
I split these to have different models for different conditions.
Regime A — Baseline Operation:
- Nominal temperature and pressure
- Efficient cooling
- Lower fault probability
- Represents stable long-term operation

In [1]:
import pandas as pd

In [2]:
data=pd.read_csv('archive/chemical_process_timeseries.csv')

In [3]:
df = data[data['operating_regime']=='A'] #filtering for the condition

In [4]:
df['operating_regime'].value_counts()

operating_regime
A    388800
Name: count, dtype: int64

In [5]:
df.drop(columns='operating_regime',inplace=True)  #deleting the column

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 388800 entries, 0 to 388799
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   timestamp             388800 non-null  str    
 1   reactor_id            388800 non-null  str    
 2   ambient_temp_effect   365600 non-null  float64
 3   reactor_temp          365285 non-null  float64
 4   reactor_pressure      365396 non-null  float64
 5   feed_flow_rate        365354 non-null  float64
 6   coolant_flow_rate     365436 non-null  float64
 7   agitator_speed_rpm    365316 non-null  float64
 8   reaction_rate         365284 non-null  float64
 9   conversion_rate       365656 non-null  float64
 10  selectivity           365085 non-null  float64
 11  yield_pct             365529 non-null  float64
 12  vibration_rms         365677 non-null  float64
 13  motor_current         365472 non-null  float64
 14  power_consumption_kw  365626 non-null  float64
 15  temp_setpoi

In [7]:
df['timestamp']= pd.to_datetime(df['timestamp'])
df['reactor_id'] = df['reactor_id'].str[-1:] #so I only have the reactornumber as id
df['reactor_id'] = df['reactor_id'].astype('int')

## Handling NaNs

In [10]:
#lets start at the top and work our way down
df['ambient_temp_effect'].sort_values() #it's the temperature effect, not the outside temperature itself, so it might be different for each of the 3 reactors. but since it only gradually changes (time-sensitive), forward/backwardfill makes sense here (but for each single reactor)
mask = df['reactor_id'] == 1 #for the first reactor
df.loc[mask, 'ambient_temp_effect'] = df.loc[mask, 'ambient_temp_effect'].ffill()

In [11]:
#reactor temperature - I would handle that similar
df.loc[mask, 'reactor_temp'] = df.loc[mask, 'reactor_temp'].interpolate(method='linear')
#but I changed ffill to interpolate: takes the mean from the next and the last entry

#reactor_pressure
df.loc[mask, 'reactor_pressure'] = df.loc[mask, 'reactor_pressure'].interpolate(method='linear')
#feed_flow_rate
df.loc[mask, 'feed_flow_rate'] = df.loc[mask, 'feed_flow_rate'].interpolate(method='linear')
#coolant_flow_rate
df.loc[mask, 'coolant_flow_rate'] = df.loc[mask, 'coolant_flow_rate'].interpolate(method='linear')
#agitator_speed_rpm
df.loc[mask, 'agitator_speed_rpm'] = df.loc[mask, 'agitator_speed_rpm'].interpolate(method='linear')
#reaction_rate
df.loc[mask, 'reaction_rate'] = df.loc[mask, 'reaction_rate'].interpolate(method='linear')
#conversion_rate
df.loc[mask, 'conversion_rate'] = df.loc[mask, 'conversion_rate'].interpolate(method='linear')
#selectivity
df.loc[mask, 'selectivity'] = df.loc[mask, 'selectivity'].interpolate(method='linear')
#yield_pct
df.loc[mask, 'yield_pct'] = df.loc[mask, 'yield_pct'].interpolate(method='linear')
#vibration_rms
df.loc[mask, 'vibration_rms'] = df.loc[mask, 'vibration_rms'].interpolate(method='linear')
#motor_current
df.loc[mask, 'motor_current'] = df.loc[mask, 'motor_current'].interpolate(method='linear')
#power_consumption_kw
df.loc[mask, 'power_consumption_kw'] = df.loc[mask, 'power_consumption_kw'].interpolate(method='linear')
#temp_setpoint
df['temp_setpoint']=df['temp_setpoint'].fillna(df['temp_setpoint'].median())#its all the same temp anyway
#pressure_setpoint
df['pressure_setpoint'] = df['pressure_setpoint'].fillna(df['pressure_setpoint'].median())#same here
#efficiency_loss_pct
df.loc[mask, 'efficiency_loss_pct'] = df.loc[mask, 'efficiency_loss_pct'].interpolate(method='linear')

In [12]:
df.columns

Index(['timestamp', 'reactor_id', 'ambient_temp_effect', 'reactor_temp',
       'reactor_pressure', 'feed_flow_rate', 'coolant_flow_rate',
       'agitator_speed_rpm', 'reaction_rate', 'conversion_rate', 'selectivity',
       'yield_pct', 'vibration_rms', 'motor_current', 'power_consumption_kw',
       'temp_setpoint', 'pressure_setpoint', 'fault_type',
       'efficiency_loss_pct', 'time_to_fault_min'],
      dtype='str')

In [13]:
#same for the other reactors but a little more handy:

columns = ['ambient_temp_effect', 'reactor_temp',
       'reactor_pressure', 'feed_flow_rate', 'coolant_flow_rate',
       'agitator_speed_rpm', 'reaction_rate', 'conversion_rate', 'selectivity',
       'yield_pct', 'vibration_rms', 'motor_current', 'power_consumption_kw',
       'temp_setpoint', 'pressure_setpoint', 'efficiency_loss_pct']
        # copied and deleted the ones I don't need here #it makes no difference if the setpoints are calculated this way or with median, so I just calculate it like the rest

#and because I can, I wrote a function for that:
def away_with_nans(reactor):
    mask = df['reactor_id'] == reactor
    for i in columns:
        df.loc[mask, i] = df.loc[mask, i].interpolate(method='linear')

#ambient temp effect is now also calculated with the mean instead of forward fill

In [14]:
away_with_nans(2)
away_with_nans(3)

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 388800 entries, 0 to 388799
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   timestamp             388800 non-null  datetime64[us]
 1   reactor_id            388800 non-null  int64         
 2   ambient_temp_effect   388799 non-null  float64       
 3   reactor_temp          388800 non-null  float64       
 4   reactor_pressure      388799 non-null  float64       
 5   feed_flow_rate        388800 non-null  float64       
 6   coolant_flow_rate     388800 non-null  float64       
 7   agitator_speed_rpm    388799 non-null  float64       
 8   reaction_rate         388800 non-null  float64       
 9   conversion_rate       388800 non-null  float64       
 10  selectivity           388800 non-null  float64       
 11  yield_pct             388800 non-null  float64       
 12  vibration_rms         388800 non-null  float64       
 13  motor_curr

In [19]:
#only time to fault left. Like I said before, I want to change it into a boolean column, since I have the time information already in the timestamp
df['normal_behavior'] = df['time_to_fault_min'].isna()

In [20]:
df

,timestamp,reactor_id,ambient_temp_effect,reactor_temp,reactor_pressure,feed_flow_rate,coolant_flow_rate,agitator_speed_rpm,reaction_rate,conversion_rate,...,yield_pct,vibration_rms,motor_current,power_consumption_kw,temp_setpoint,pressure_setpoint,fault_type,efficiency_loss_pct,time_to_fault_min,normal_behavior
0,2024-01-01 00:00:00,1,0.000000e+00,181.135558,15.791013,101.108882,79.154645,305.779931,0.724542,99.151760,...,82.032893,1.470297,45.882315,41.294083,180.0,12.0,0,0.0,NaN,True
1,2024-01-01 00:01:00,1,4.848174e-04,182.249820,15.706975,98.932369,79.874191,302.283603,0.728999,98.638957,...,81.301503,1.425847,46.755905,42.080314,180.0,12.0,0,0.0,NaN,True
2,2024-01-01 00:02:00,1,9.696348e-04,183.129737,15.593397,99.792219,80.593736,305.814440,0.732519,99.154925,...,82.006626,1.448190,46.309038,41.678134,180.0,12.0,0,0.0,NaN,True
3,2024-01-01 00:03:00,1,1.454452e-03,183.020987,15.527333,99.854222,80.190444,304.436123,0.732084,98.994864,...,81.562261,1.473448,45.862171,41.275954,180.0,12.0,0,0.0,NaN,True
4,2024-01-01 00:04:00,1,1.939270e-03,183.077096,15.554044,99.302349,78.571707,304.811418,0.732308,99.252000,...,82.411124,1.523491,46.880227,42.192204,180.0,12.0,0,0.0,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
388795,2024-03-30 23:55:00,3,-1.939270e-03,180.512489,15.698175,98.294404,79.056576,294.996530,0.722050,98.628870,...,81.132708,1.482459,45.609944,42.239260,180.0,12.0,0,0.0,NaN,True
388796,2024-03-30 23:56:00,3,-1.454452e-03,180.338679,15.717705,98.184695,82.652110,299.038660,0.721355,99.004749,...,81.402088,1.507740,45.391636,40.852472,180.0,12.0,0,0.0,NaN,True
388797,2024-03-30 23:57:00,3,-9.696348e-04,179.600059,15.700481,99.075055,80.572876,306.286187,0.718400,98.894388,...,81.457731,1.533022,46.423420,41.781078,180.0,12.0,0,0.0,NaN,True
388798,2024-03-30 23:58:00,3,-4.848174e-04,180.023226,15.650709,100.313890,81.067978,296.455683,0.720093,98.566737,...,81.293929,1.447584,45.742744,41.168469,180.0,12.0,0,0.0,NaN,True


# Preprocessing for the stressed condition
Regime B — Stress Operation
- Higher temperature and pressure
- Reduced cooling efficiency
- Increased sensor noise
- Higher fault probability
- Represents harsh or high-throughput conditions

same thing with the stressed data

In [21]:
df_stressed = data[data['operating_regime']=='B'] #filtering for the condition
df_stressed['operating_regime'].value_counts()

operating_regime
B    388800
Name: count, dtype: int64

In [22]:
df_stressed.drop(columns='operating_regime', inplace=True)  #deleting the column
df_stressed.info()

<class 'pandas.DataFrame'>
RangeIndex: 388800 entries, 388800 to 777599
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   timestamp             388800 non-null  str    
 1   reactor_id            388800 non-null  str    
 2   ambient_temp_effect   365439 non-null  float64
 3   reactor_temp          365350 non-null  float64
 4   reactor_pressure      365061 non-null  float64
 5   feed_flow_rate        365486 non-null  float64
 6   coolant_flow_rate     365519 non-null  float64
 7   agitator_speed_rpm    365509 non-null  float64
 8   reaction_rate         365486 non-null  float64
 9   conversion_rate       365528 non-null  float64
 10  selectivity           365306 non-null  float64
 11  yield_pct             365547 non-null  float64
 12  vibration_rms         365464 non-null  float64
 13  motor_current         365511 non-null  float64
 14  power_consumption_kw  365480 non-null  float64
 15  temp_s

In [23]:
df_stressed['timestamp']= pd.to_datetime(df_stressed['timestamp'])
df_stressed['reactor_id'] = df_stressed['reactor_id'].str[-1:] #so I only have the reactornumber as id
df_stressed['reactor_id'] = df_stressed['reactor_id'].astype('int')

In [25]:
df_stressed['reactor_id'].value_counts()

reactor_id
1    129600
2    129600
3    129600
Name: count, dtype: int64

In [26]:
#have to define it with the stressed dataset. If I had more datasets, I could use that as an input, but for just two, this is also sufficient
def away_with_nans_stressed(reactor):
    mask = df_stressed['reactor_id'] == reactor
    for i in columns:
        df_stressed.loc[mask, i] = df_stressed.loc[mask, i].interpolate(method='linear')
        ##was claude unten sagt. damit auch die nans ganz am ende und ganz am anfang weg sind

away_with_nans_stressed(1)
away_with_nans_stressed(2)
away_with_nans_stressed(3)

In [ ]:
'''cols = df_stressed.columns.tolist()
idx = cols.index(i)

has_next = idx + 1 < len(cols)
has_prev = idx - 1 >= 0

if df_stressed.loc[mask, i].isna().any():
    if has_next and df_stressed.loc[mask, cols[idx+1]].isna().any():
        df_stressed.loc[mask, i] = df_stressed.loc[mask, i].ffill()
    elif has_prev and df_stressed.loc[mask, cols[idx-1]].isna().any():
        df_stressed.loc[mask, i] = df_stressed.loc[mask, i].bfill()'''

In [27]:
df_stressed['normal_behavior'] = df_stressed['time_to_fault_min'].isna()
df_stressed.info()

<class 'pandas.DataFrame'>
RangeIndex: 388800 entries, 388800 to 777599
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   timestamp             388800 non-null  datetime64[us]
 1   reactor_id            388800 non-null  int64         
 2   ambient_temp_effect   388800 non-null  float64       
 3   reactor_temp          388800 non-null  float64       
 4   reactor_pressure      388800 non-null  float64       
 5   feed_flow_rate        388800 non-null  float64       
 6   coolant_flow_rate     388800 non-null  float64       
 7   agitator_speed_rpm    388800 non-null  float64       
 8   reaction_rate         388800 non-null  float64       
 9   conversion_rate       388800 non-null  float64       
 10  selectivity           388800 non-null  float64       
 11  yield_pct             388800 non-null  float64       
 12  vibration_rms         388800 non-null  float64       
 13  motor

# exporting the data

In [28]:
df.to_csv('archive/baseline_cleaned.csv', index=False)
df_stressed.to_csv('archive/stressed_cleaned.csv', index=False)